# Step 3 — Live UnifoLM Wipe Success (Layer F)

**Timing-only** dual-process lives in `step3_dual_thread_mujoco.ipynb` (Layer D).
This notebook is the missing **task-success** companion:

| | Layer D (`step3_dual_thread`) | Layer F (this notebook) |
|---|---|---|
| VLA | live UnifoLM | live UnifoLM |
| Metrics | step ms / Hz | **grasp / contact / coverage / task_success** + timing |
| Cloth | static scene | interactive mocap cloth |
| Gripper | n/a | proximity-synthetic (UnifoLM EE has no Dex1 channel) |

Not dataset-oracle (that is Step 4). Do **not** conflate these success rates.

```bash
export HF_HOME=/raid/data/aihimekpen/hf_cache
python3 -m src.step3_live_wipe_eval --bridge esn --duration_s 30 --record_video
python3 -m src.step3_live_wipe_eval --bridges esn,zoh --duration_s 20
```


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("HF_HOME", "/raid/data/aihimekpen/hf_cache")
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/raid/data/aihimekpen/hf_cache/hub")

NOTEBOOK_DIR = Path.cwd().resolve()
RESEARCH_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
RESEARCH_DIR = RESEARCH_DIR.resolve()
os.chdir(RESEARCH_DIR)
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

print(f"Research root : {RESEARCH_DIR}")
print(f"Results go to : {RESEARCH_DIR / 'results' / 'step3_live_wipe'}")
print(f"HF_HOME       : {os.environ.get('HF_HOME')}")

In [ ]:
MOCK = False                 # False = live UnifoLM (paper)
BRIDGES = ["esn", "zoh"]     # compare ESN vs ZOH under live VLA
DURATION_S = 30.0            # after VLA load; max 30s
RECORD_VIDEO = True          # one MP4 for first bridge only
INIT_EPISODE = 160           # seed G1 pose from held-out wipe demo
INSTRUCTION = "Wipe the table with the cloth."
CONTROL_HZ = 100.0
VLA_HZ = 2.0
DEVICE = "cuda"

print(f"MOCK={MOCK} | BRIDGES={BRIDGES} | duration={DURATION_S}s | video={RECORD_VIDEO}")

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from src.paths import results_path
from src.step3_dual_thread_mujoco import resolve_esn_checkpoint, resolve_mjcf_path
from src.step3_live_wipe_eval import LiveWipeConfig, LiveWipeController, print_live_wipe_summary

if not MOCK and not torch.cuda.is_available():
    raise RuntimeError("CUDA required for live UnifoLM.")

mjcf = resolve_mjcf_path(None)
ckpt = resolve_esn_checkpoint(None)
out_dir = results_path("step3_live_wipe")
out_dir.mkdir(parents=True, exist_ok=True)

reports = []
for i, bridge in enumerate(BRIDGES):
    record_video = bool(RECORD_VIDEO) and i == 0
    tag = "mock" if MOCK else "live"
    video_path = out_dir / f"live_wipe_{bridge}_{tag}.mp4"
    cfg = LiveWipeConfig(
        mjcf_path=mjcf,
        esn_checkpoint=str(ckpt),
        mock=MOCK,
        duration_s=DURATION_S,
        control_hz=CONTROL_HZ,
        vla_hz=VLA_HZ,
        instruction=INSTRUCTION,
        device=DEVICE,
        record_video=record_video,
        video_path=video_path if record_video else None,
        init_episode=INIT_EPISODE,
        bridge=bridge,
    )
    stats = LiveWipeController(cfg).run()
    report = stats.to_dict()
    report["esn_checkpoint"] = str(ckpt)
    ep_path = out_dir / f"live_wipe_report_{bridge}_{tag}.json"
    ep_path.write_text(json.dumps(report, indent=2))
    reports.append(report)
    print_live_wipe_summary(stats, report_path=ep_path)

summary = {
    "n_trials": len(reports),
    "mock_vla": MOCK,
    "duration_s": DURATION_S,
    "bridges": BRIDGES,
    "gripper_mode": "proximity_synthetic",
    "grasp_success_rate": float(np.mean([
        1.0 if (r.get("task_metrics") or {}).get("grasp_success") else 0.0 for r in reports
    ])),
    "task_success_rate": float(np.mean([
        1.0 if (r.get("task_metrics") or {}).get("task_success") else 0.0 for r in reports
    ])),
    "table_contact_ratio_mean": float(np.mean([
        float((r.get("task_metrics") or {}).get("table_contact_ratio", 0.0)) for r in reports
    ])),
    "reports": reports,
}
sum_path = out_dir / f"live_wipe_summary_{'mock' if MOCK else 'live'}.json"
sum_path.write_text(json.dumps(summary, indent=2))
(out_dir / "live_wipe_report.json").write_text(json.dumps(summary, indent=2))

rows = []
for r in reports:
    tm = r.get("task_metrics") or {}
    rows.append({
        "bridge": r["bridge"],
        "mock_vla": r["mock_vla"],
        "mean_step_ms": r["mean_step_ms"],
        "achieved_hz": r["achieved_control_hz"],
        "vla_ticks": r["vla_ticks"],
        "grasp_success": tm.get("grasp_success"),
        "task_success": tm.get("task_success"),
        "wipe_path_m": tm.get("wipe_path_length_m"),
        "table_contact_ratio": tm.get("table_contact_ratio"),
        "wipe_coverage_m2": tm.get("wipe_coverage_m2"),
        "video_path": r.get("video_path"),
    })
df = pd.DataFrame(rows)
csv_path = out_dir / f"live_wipe_summary_{'mock' if MOCK else 'live'}.csv"
df.to_csv(csv_path, index=False)
display(df)
print(f"Summary: {sum_path}")
print(
    f"grasp={summary['grasp_success_rate']:.1%} | "
    f"contact={summary['table_contact_ratio_mean']:.1%} | "
    f"task_success={summary['task_success_rate']:.1%}"
)

In [ ]:
from IPython.display import Video, display, Markdown
from pathlib import Path

out_dir = Path("results/step3_live_wipe")
vids = sorted(out_dir.glob("live_wipe_*.mp4"))
if not vids:
    display(Markdown("_No MP4 yet — re-run with RECORD_VIDEO=True._"))
else:
    for v in vids:
        display(Markdown(f"**{v.name}** ({v.stat().st_size/1e6:.2f} MB)"))
        display(Video(str(v), embed=True, width=640))